# **Step 1: Importing libraries and data**

In [ ]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import time
import os



CPI_IRE_PATH  = "/content/drive/MyDrive/Colab Notebooks/Raw Datasets/CPI Ireland.xlsx"
CPI_USA_PATH  = "/content/drive/MyDrive/Colab Notebooks/Raw Datasets/Consumer price index US.csv"
NER_PATH = "/content/drive/MyDrive/Colab Notebooks/Raw Datasets/NER.csv"
OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/Cleaned Datasets"

START_YEAR = 2015
END_YEAR   = 2025

# **Step 2: Cleaning Ireland's CPI**

In [ ]:
print("SECTION 1: Cleaning Ireland CPI")
print("=" * 60)

# 1.1 Loading the datasets
df_ire = pd.read_excel(CPI_IRE_PATH, sheet_name="Monthly")
print(f"\n[Raw] Shape: {df_ire.shape}")
print(f"[Raw] Columns: {list(df_ire.columns)}")
print(f"[Raw] First 3 rows:\n{df_ire.head(3)}")
print(f"[Raw] Last 3 rows:\n{df_ire.tail(3)}")

#1.2 Renaming columns for understanding
df_ire.columns = ["observation_date", "CPI_Ireland"]
print(f"\n[Step] Renamed columns → {list(df_ire.columns)}")

#1.3 Parsing dates
df_ire["observation_date"] = pd.to_datetime(df_ire["observation_date"])
print(f"[Step] Parsed 'observation_date' as datetime")

#1.4 Check & handle missing values
missing = df_ire["CPI_Ireland"].isna().sum()
print(f"[Check] Missing values in CPI_Ireland: {missing}")
if missing > 0:
    df_ire["CPI_Ireland"] = df_ire["CPI_Ireland"].interpolate(method="linear")
    print(f"  → Filled {missing} actual missing values using linear interpolation")

#1.5 Checking for duplicates
dupes = df_ire.duplicated(subset="observation_date").sum()
print(f"[Check] Duplicate dates: {dupes}")
if dupes > 0:
    df_ire = df_ire.drop_duplicates(subset="observation_date", keep="first")
    print(f"  → Dropped {dupes} duplicate rows")

#1.6 Checking data types and value range
print(f"\nSTEP 6 - Checking the data type and it's value range")
print(f"  CPI_Ireland data type : {df_ire['CPI_Ireland'].dtype}")
print(f"  Minimum value         : {df_ire['CPI_Ireland'].min():.2f}")
print(f"  Maximum value         : {df_ire['CPI_Ireland'].max():.2f}")

#1.7 Filtering from 2015-2025
mask = (df_ire["observation_date"].dt.year >= START_YEAR) & \
       (df_ire["observation_date"].dt.year <= END_YEAR)
df_ire = df_ire[mask].reset_index(drop=True)
print(f"[Step] Filtered to {START_YEAR}–{END_YEAR} → {len(df_ire)} rows")
#1.8 Outlier Detection
print(f"\n[Outlier Check] Detecting outliers in CPI_Ireland using IQR method")
Q1  = df_ire["CPI_Ireland"].quantile(0.25)   # 25th percentile
Q3  = df_ire["CPI_Ireland"].quantile(0.75)   # 75th percentile
IQR = Q3 - Q1                                 # interquartile range
lower_fence = Q1 - 1.5 * IQR                 # lower outlier boundary
upper_fence = Q3 + 1.5 * IQR                 # upper outlier boundary
print(f"  Q1 (25th percentile) : {Q1:.2f}")
print(f"  Q3 (75th percentile) : {Q3:.2f}")
print(f"  IQR                  : {IQR:.2f}")
print(f"  Lower fence          : {lower_fence:.2f}")
print(f"  Upper fence          : {upper_fence:.2f}")

# Find any rows which fall outside the fences
outliers_ire = df_ire[
    (df_ire["CPI_Ireland"] < lower_fence) |
    (df_ire["CPI_Ireland"] > upper_fence)
]
print(f"  Outliers found       : {len(outliers_ire)}")
if not outliers_ire.empty:
    print(f"  Outlier rows:\n{outliers_ire}")
else:
    print(f"  → No outliers detected, data looks clean")

#1.9 Creating a clean period key
df_ire["date"] = df_ire["observation_date"].dt.to_period("M").astype(str)

#1.10 Keeping only necessary columns
df_ire_clean = df_ire[["date", "CPI_Ireland"]].copy()

#1.11 Rounding upto 2 decimal points
df_ire_clean["CPI_Ireland"] = df_ire_clean["CPI_Ireland"].round(2)

# 1.12 Final checks
print(f"\n[Clean] Final shape: {df_ire_clean.shape}")
print(f"[Clean] Sample:\n{df_ire_clean.head(3)}")
print(f"[Clean] Missing values:\n{df_ire_clean.isna().sum().to_string()}")

#1.13 Saving
out_path = os.path.join(OUTPUT_DIR, "cleaned_cpi_ireland.csv")
df_ire_clean.to_csv(out_path, index=False)
print(f"\n✓ Saved → {out_path}")

SECTION 1: Cleaning Ireland CPI

[Raw] Shape: (360, 2)
[Raw] Columns: ['observation_date', 'CP0000IEM086NEST']
[Raw] First 3 rows:
  observation_date  CP0000IEM086NEST
0       1996-01-01              68.1
1       1996-02-01              68.6
2       1996-03-01              68.9
[Raw] Last 3 rows:
    observation_date  CP0000IEM086NEST
357       2025-10-01             123.0
358       2025-11-01             122.7
359       2025-12-01             123.4

[Step] Renamed columns → ['observation_date', 'CPI_Ireland']
[Step] Parsed 'observation_date' as datetime
[Check] Missing values in CPI_Ireland: 0
[Check] Duplicate dates: 0

STEP 6 - Checking the data type and it's value range
  CPI_Ireland data type : float64
  Minimum value         : 68.10
  Maximum value         : 123.40
[Step] Filtered to 2015–2025 → 132 rows

[Outlier Check] Detecting outliers in CPI_Ireland using IQR method
  Q1 (25th percentile) : 100.40
  Q3 (75th percentile) : 116.98
  IQR                  : 16.58
  Lower fence  

# **Step 3: Cleaning USA CPI**

In [ ]:
print("SECTION 2: Cleaning USA CPI")

#2.1 Loading
df_usa = pd.read_csv(CPI_USA_PATH)
print(f"\n[Raw] Shape: {df_usa.shape}")
print(f"[Raw] Columns: {list(df_usa.columns)}")
print(f"[Raw] First 3 rows:\n{df_usa.head(3)}")
print(f"[Raw] Last 3 rows:\n{df_usa.tail(3)}")

#2.2 Renaming the columns
df_usa.columns = ["observation_date", "CPI_USA_raw"]
print(f"\n[Step] Renamed columns → {list(df_usa.columns)}")

#2.3 Parsing dates
df_usa["observation_date"] = pd.to_datetime(df_usa["observation_date"])
print(f"[Step] Parsed 'observation_date' as datetime")

#2.4 Checking & handling missing values
missing = df_usa["CPI_USA_raw"].isna().sum()
print(f"[Check] Missing values in CPI_USA_raw: {missing}")
if missing > 0:
    df_usa["CPI_USA_raw"] = df_usa["CPI_USA_raw"].interpolate(method="linear")
    print(f"  → Filled {missing} missing values using linear interpolation")

#2.5 Check for all the duplicates
dupes = df_usa.duplicated(subset="observation_date").sum()
print(f"[Check] Duplicate dates: {dupes}")
if dupes > 0:
    df_usa = df_usa.drop_duplicates(subset="observation_date", keep="first")

#2.6 Filtering specifically from 2015–2025
mask = (df_usa["observation_date"].dt.year >= START_YEAR) & \
       (df_usa["observation_date"].dt.year <= END_YEAR)
df_usa_full = df_usa.copy()           # keep full series for rebasing
df_usa = df_usa[mask].reset_index(drop=True)
print(f"[Step] Filtered to {START_YEAR}–{END_YEAR} → {len(df_usa)} rows")

#2.7 Rebasing CPI to 2015 = 100
# Calculating mean of 2015 monthly values
base_vals = df_usa_full[df_usa_full["observation_date"].dt.year == 2015]["CPI_USA_raw"]
base_2015 = base_vals.mean()
print(f"[Step] 2015 average CPI_USA (1982-84=100 base): {base_2015:.4f}")
df_usa["CPI_USA"] = (df_usa["CPI_USA_raw"] / base_2015 * 100).round(2)
print(f"[Step] Rebased to 2015=100. New range: "
      f"{df_usa['CPI_USA'].min():.2f} – {df_usa['CPI_USA'].max():.2f}")
#2.8 Outlier Detection
print(f"\n[Outlier Check] Detecting outliers in CPI_USA using IQR method")
Q1  = df_usa["CPI_USA"].quantile(0.25)
Q3  = df_usa["CPI_USA"].quantile(0.75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR
print(f"  Q1 (25th percentile) : {Q1:.2f}")
print(f"  Q3 (75th percentile) : {Q3:.2f}")
print(f"  IQR                  : {IQR:.2f}")
print(f"  Lower fence          : {lower_fence:.2f}")
print(f"  Upper fence          : {upper_fence:.2f}")

outliers_usa = df_usa[
    (df_usa["CPI_USA"] < lower_fence) |
    (df_usa["CPI_USA"] > upper_fence)
]
print(f"  Outliers found       : {len(outliers_usa)}")
if not outliers_usa.empty:
    print(f"  Outlier rows:\n{outliers_usa}")
else:
    print(f"  → No outliers detected, data looks clean")

#2.9 Creating the period key
df_usa["date"] = df_usa["observation_date"].dt.to_period("M").astype(str)

#2.10 Keeping only necessary columns
df_usa_clean = df_usa[["date", "CPI_USA"]].copy()

#2.11 Final checks
print(f"\n[Clean] Final shape: {df_usa_clean.shape}")
print(f"[Clean] Sample:\n{df_usa_clean.head(3)}")
print(f"[Clean] Missing values:\n{df_usa_clean.isna().sum().to_string()}")

#2.12 Saving
out_path = os.path.join(OUTPUT_DIR, "cleaned_cpi_usa.csv")
df_usa_clean.to_csv(out_path, index=False)
print(f"\n✓ Saved → {out_path}")

SECTION 2: Cleaning USA CPI

[Raw] Shape: (120, 2)
[Raw] Columns: ['observation_date', 'CPIAUCSL']
[Raw] First 3 rows:
  observation_date  CPIAUCSL
0       01-01-2016   237.652
1       01-02-2016   237.336
2       01-03-2016   238.080
[Raw] Last 3 rows:
    observation_date  CPIAUCSL
117       01-10-2025       NaN
118       01-11-2025   325.063
119       01-12-2025   326.031

[Step] Renamed columns → ['observation_date', 'CPI_USA_raw']
[Step] Parsed 'observation_date' as datetime
[Check] Missing values in CPI_USA_raw: 1
  → Filled 1 missing values using linear interpolation
[Check] Duplicate dates: 0
[Step] Filtered to 2015–2025 → 120 rows
[Step] 2015 average CPI_USA (1982-84=100 base): nan
[Step] Rebased to 2015=100. New range: nan – nan

[Outlier Check] Detecting outliers in CPI_USA using IQR method
  Q1 (25th percentile) : nan
  Q3 (75th percentile) : nan
  IQR                  : nan
  Lower fence          : nan
  Upper fence          : nan
  Outliers found       : 0
  → No outliers

# **Step 4: Loading Nominal exchange rate from pre-downloaded CSV**

In [ ]:
print("SECTION 3: Loading nominal exchange rate (EUR/USD) from cleaned_ner.csv")
print("=" * 60)

df_ner = pd.read_csv(NER_PATH)
print(f"\n[Load] Shape: {df_ner.shape}")
print(f"[Load] Sample:\n{df_ner.head(3)}")

# Check for missing values
missing = df_ner["NER"].isna().sum()
print(f"[Check] Missing nominal exchange rate values: {missing}")

# Check for duplicates
dupes = df_ner.duplicated(subset="date").sum()
print(f"[Check] Duplicate date entries: {dupes}")

# Validating the rate range
print(f"[Check] Nominal exchange rate  range: {df_ner['NER'].min():.4f} – {df_ner['NER'].max():.4f}")
# Outlier Detection
print(f"\n[Outlier Check] Detecting outliers in NER using IQR method")
Q1  = df_ner["NER"].quantile(0.25)
Q3  = df_ner["NER"].quantile(0.75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR
print(f"  Q1 (25th percentile) : {Q1:.4f}")
print(f"  Q3 (75th percentile) : {Q3:.4f}")
print(f"  IQR                  : {IQR:.4f}")
print(f"  Lower fence          : {lower_fence:.4f}")
print(f"  Upper fence          : {upper_fence:.4f}")

outliers_ner = df_ner[
    (df_ner["NER"] < lower_fence) |
    (df_ner["NER"] > upper_fence)
]
print(f"  Outliers found       : {len(outliers_ner)}")
if not outliers_ner.empty:
    print(f"  Outlier rows:\n{outliers_ner}")
else:
    print(f"  → No outliers detected, data looks clean")

# Rounding up to 3decimal places
df_ner["NER"] = df_ner["NER"].round(3)
# Saving
out_path = os.path.join(OUTPUT_DIR, "cleaned_ner.csv")
df_ner.to_csv(out_path, index=False)
print(f"\n✓ Saved → {out_path}")

print("\n" + "=" * 60)
print("CLEANING COMPLETE — 3 files saved:")
print("  cleaned_cpi_ireland.csv")
print("  cleaned_cpi_usa.csv")
print("  cleaned_ner.csv")
print("=" * 60)

SECTION 3: Loading nominal exchange rate (EUR/USD) from cleaned_ner.csv

[Load] Shape: (132, 2)
[Load] Sample:
      date     NER
0  2015-01  1.1291
1  2015-02  1.1340
2  2015-03  1.0822
[Check] Missing nominal exchange rate values: 0
[Check] Duplicate date entries: 0
[Check] Nominal exchange rate  range: 0.9904 – 1.2316

[Outlier Check] Detecting outliers in NER using IQR method
  Q1 (25th percentile) : 1.0861
  Q3 (75th percentile) : 1.1577
  IQR                  : 0.0716
  Lower fence          : 0.9786
  Upper fence          : 1.2651
  Outliers found       : 0
  → No outliers detected, data looks clean

✓ Saved → /content/drive/MyDrive/Colab Notebooks/Cleaned Datasets/cleaned_ner.csv

CLEANING COMPLETE — 3 files saved:
  cleaned_cpi_ireland.csv
  cleaned_cpi_usa.csv
  cleaned_ner.csv


# **Step 5: Importing Cleaned datasets**

In [ ]:
# Step 1: Load WITHOUT names and header
df_ie = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Cleaned Datasets/cleaned_cpi_ireland.csv')
df_us = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Cleaned Datasets/cleaned_cpi_usa.csv')
df_ner = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Cleaned Datasets/cleaned_ner.csv')

print("Raw Ireland columns:", df_ie.columns.tolist())
print("Raw Ireland head:\n", df_ie.head())

Raw Ireland columns: ['date', 'CPI_Ireland']
Raw Ireland head:
       date  CPI_Ireland
0  2015-01         98.7
1  2015-02         99.3
2  2015-03        100.0
3  2015-04         99.9
4  2015-05        100.4


# **Step 6: Setting the date column as index**

In [ ]:
df_ie = df_ie.set_index('date')['CPI_Ireland'].astype(float)
df_us = df_us.set_index('date')['CPI_USA'].astype(float)
df_ner = df_ner.set_index('date')['NER'].astype(float)

# Output
print("Sample after indexing")
print("\nIreland Series head:\n", df_ie.head())
print("USA Series head:\n", df_us.head())
print("NER Series head:\n", df_ner.head())

Sample after indexing

Ireland Series head:
 date
2015-01     98.7
2015-02     99.3
2015-03    100.0
2015-04     99.9
2015-05    100.4
Name: CPI_Ireland, dtype: float64
USA Series head:
 date
2016-01   NaN
2016-01   NaN
2016-01   NaN
2016-01   NaN
2016-01   NaN
Name: CPI_USA, dtype: float64
NER Series head:
 date
2015-01    1.129
2015-02    1.134
2015-03    1.082
2015-04    1.074
2015-05    1.114
Name: NER, dtype: float64


# **Step 7: Drop duplicates if any**

In [ ]:
df_ie = df_ie[~df_ie.index.duplicated(keep='first')]
df_us = df_us[~df_us.index.duplicated(keep='first')]
df_ner = df_ner[~df_ner.index.duplicated(keep='first')]

print("After droppping duplicates")
print("\nShapes - IE:", df_ie.shape, "US:", df_us.shape, "NER:", df_ner.shape)

After droppping duplicates

Shapes - IE: (132,) US: (10,) NER: (132,)


# **Step 8: Merge to DataFrame**

In [ ]:
df = pd.concat([df_ie, df_us, df_ner], axis=1, join='inner', sort=False)

print("Dataset after concatenation:")
print("\nMerged shape:", df.shape)
print("Merged head:\n", df.head())
print("Columns:", df.columns.tolist())

Dataset after concatenation:

Merged shape: (10, 3)
Merged head:
          CPI_Ireland  CPI_USA    NER
date                                
2016-01         98.7      NaN  1.086
2017-01         98.9      NaN  1.062
2018-01         99.2      NaN  1.203
2019-01        100.0      NaN  1.141
2020-01        101.1      NaN  1.109
Columns: ['CPI_Ireland', 'CPI_USA', 'NER']


# **Step 9: Calculate RER**

In [ ]:
print("Calulating the real exchange rate")
df['RER'] = (df['NER'] * (df['CPI_USA'] / df['CPI_Ireland'])).round(3)
print("RER sample:", df['RER'].head())

Calulating the real exchange rate
RER sample: date
2016-01   NaN
2017-01   NaN
2018-01   NaN
2019-01   NaN
2020-01   NaN
Name: RER, dtype: float64


# **Step 10: Filter and save merged dataset**

In [ ]:
outputDir = "/content/drive/MyDrive/Colab Notebooks/Merged Datasets"
df.to_csv('merged_ner_data.csv')
out_path = os.path.join(outputDir, "merged_ner_data.csv")
df.to_csv(out_path, index=False)
print(f"Saved to {out_path}")
print("\nFinal head:\n", df.head())
print("Saved to merged_rer_data.csv")

Saved to /content/drive/MyDrive/Colab Notebooks/Merged Datasets/merged_ner_data.csv

Final head:
          CPI_Ireland  CPI_USA    NER  RER
date                                     
2016-01         98.7      NaN  1.086  NaN
2017-01         98.9      NaN  1.062  NaN
2018-01         99.2      NaN  1.203  NaN
2019-01        100.0      NaN  1.141  NaN
2020-01        101.1      NaN  1.109  NaN
Saved to merged_rer_data.csv
